# NB11 — AutoGluon Extended Time Budget Test

## Objective

This notebook evaluates whether increasing the AutoGluon training budget improves forecasting performance enough to justify the additional computational cost.

NB9 used AutoGluon Tabular with:

- 2 walk-forward validation folds
- 600 seconds per fold
- advanced_v1 feature set
- full training history

NB11 keeps the same validated setup but increases the AutoGluon time budget to 3600 seconds on the most recent validation fold.

The goal is not to change the pipeline, but to answer a practical production question:

> Does a longer AutoML search produce a meaningful improvement compared with the 600-second benchmark?

This is a cost-benefit experiment.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
from autogluon.tabular import TabularPredictor

sys.path.append(str(Path().resolve().parent))

from src.features import add_baseline_features, add_advanced_features, encode_family
from src.metrics import evaluate_regression
from src.split import build_walk_forward_folds
from src.automl import evaluate_autogluon

## Setup

NB11 uses a separate artifact directory to avoid overwriting NB9 predictors.

Only one variable changes compared with NB9:

- AutoGluon time limit: from 600s to 3600s

All other components remain unchanged.

In [2]:
SEED = 42

DATA_DIR = Path("../data")

ARTIFACTS_DIR = Path("../artifacts/nb11_autogluon_extended_budget")
AUTOGLUON_DIR = ARTIFACTS_DIR / "autogluon_models"

NB9_ARTIFACTS_DIR = Path("../artifacts/nb9_autogluon_tabular")
NB9_METRICS_PATH = NB9_ARTIFACTS_DIR / "nb9_autogluon_fold_metrics.csv"
NB9_SUMMARY_PATH = NB9_ARTIFACTS_DIR / "nb9_autogluon_summary.csv"
NB9_LEADERBOARD_PATH = NB9_ARTIFACTS_DIR / "nb9_autogluon_leaderboard.csv"

MLFLOW_TRACKING_URI = "file:../mlruns"
MLFLOW_EXPERIMENT_NAME = "store_sales_forecasting"

N_FOLDS = 2
VAL_SIZE = 28

AUTOGLUON_TIME_LIMIT = 3600
AUTOGLUON_PRESETS = "medium_quality"

FORCE_AUTOML_RETRAIN = False

METRICS_PATH = ARTIFACTS_DIR / "nb11_autogluon_extended_budget_metrics.csv"
SUMMARY_PATH = ARTIFACTS_DIR / "nb11_autogluon_extended_budget_summary.csv"
LEADERBOARD_PATH = ARTIFACTS_DIR / "nb11_autogluon_extended_budget_leaderboard.csv"
FOLDS_PATH = ARTIFACTS_DIR / "nb11_validation_fold.csv"

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
AUTOGLUON_DIR.mkdir(parents=True, exist_ok=True)

print("Setup OK")
print("Data dir:", DATA_DIR.resolve())
print("Artifacts dir:", ARTIFACTS_DIR.resolve())
print("AutoGluon time limit:", AUTOGLUON_TIME_LIMIT)

Setup OK
Data dir: /home/donatocorbacio/projects/store-sales-project/data
Artifacts dir: /home/donatocorbacio/projects/store-sales-project/artifacts/nb11_autogluon_extended_budget
AutoGluon time limit: 3600


## MLflow Setup

In [3]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

print("Active experiment:", MLFLOW_EXPERIMENT_NAME)

Active experiment: store_sales_forecasting


## Load Data

In [4]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["date"])
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["date"])

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Train range:", train["date"].min(), "->", train["date"].max())
print("Test range:", test["date"].min(), "->", test["date"].max())

display(train.head())

Train shape: (3000888, 6)
Test shape: (28512, 5)
Train range: 2013-01-01 00:00:00 -> 2017-08-15 00:00:00
Test range: 2017-08-16 00:00:00 -> 2017-08-31 00:00:00


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


## Rebuild Validated Feature Pipeline

The same `advanced_v1` feature pipeline used in NB5, NB7, NB9 and NB10 is reused here.

No new features are introduced in NB11.

In [5]:
train = train.sort_values(["store_nbr", "family", "date"]).copy()
test = test.sort_values(["store_nbr", "family", "date"]).copy()

for df in [train, test]:
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    df["dayofweek"] = df["date"].dt.dayofweek
    df["weekofyear"] = df["date"].dt.isocalendar().week.astype(int)
    df["is_weekend"] = df["dayofweek"].isin([5, 6]).astype(int)

advanced_train = add_baseline_features(train)
advanced_train = add_advanced_features(advanced_train)
advanced_train = advanced_train.dropna().copy()

advanced_train, advanced_test, family_mapping = encode_family(advanced_train, test)

advanced_features = [
    "store_nbr",
    "family",
    "onpromotion",
    "year",
    "month",
    "day",
    "dayofweek",
    "weekofyear",
    "is_weekend",
    "lag_1",
    "lag_7",
    "rolling_mean_7",
    "rolling_std_7",
    "rolling_mean_14",
    "trend_1_7",
    "promo_last_7",
]

print("Advanced train shape:", advanced_train.shape)
print("Advanced date range:", advanced_train["date"].min(), "->", advanced_train["date"].max())
print("Number of features:", len(advanced_features))

Advanced train shape: (2975940, 19)
Advanced date range: 2013-01-15 00:00:00 -> 2017-08-15 00:00:00
Number of features: 16


## Build Validation Fold

NB9 used two recent walk-forward folds.

NB11 uses only the most recent fold as an extended-budget pilot.

This keeps the experiment practical while still testing the most business-relevant validation window.

In [6]:
all_folds = build_walk_forward_folds(
    df=advanced_train,
    n_folds=N_FOLDS,
    val_size=VAL_SIZE,
)

folds_df = pd.DataFrame(all_folds)
display(folds_df)

selected_folds = [all_folds[-1]]
selected_folds_df = pd.DataFrame(selected_folds)

display(selected_folds_df)

selected_folds_df.to_csv(FOLDS_PATH, index=False)

for fold_info in selected_folds:
    print(
        f"Selected fold {fold_info['fold']} | "
        f"train <= {pd.Timestamp(fold_info['train_end']).date()} | "
        f"val: {pd.Timestamp(fold_info['val_start']).date()} -> "
        f"{pd.Timestamp(fold_info['val_end']).date()}"
    )

,fold,train_end,val_start,val_end
0,1,2017-06-20,2017-06-21,2017-07-18
1,2,2017-07-18,2017-07-19,2017-08-15


,fold,train_end,val_start,val_end
0,2,2017-07-18,2017-07-19,2017-08-15


Selected fold 2 | train <= 2017-07-18 | val: 2017-07-19 -> 2017-08-15


## 6. Run AutoGluon Extended Budget Benchmark

This cell runs AutoGluon with 3600 seconds on the selected fold.

If cached artifacts already exist and `FORCE_AUTOML_RETRAIN=False`, the notebook loads the previous NB11 results instead of retraining.

In [7]:
with mlflow.start_run(run_name="benchmark_autogluon_tabular_extended_budget_nb11"):

    mlflow.set_tags({
        "project": "store_sales_forecasting",
        "notebook": "nb11",
        "experiment_type": "automl_extended_budget",
        "benchmark_type": "autogluon_tabular",
        "validation_strategy": "walk_forward_single_recent_fold",
        "dataset_scope": "full_data",
        "feature_set": "advanced_v1",
        "production_candidate": "false",
        "model_registry": "false",
    })

    mlflow.log_params({
        "n_folds": len(selected_folds),
        "val_size_days": VAL_SIZE,
        "n_features": len(advanced_features),
        "feature_set": "advanced_v1",
        "target": "sales",
        "autogluon_model": "TabularPredictor",
        "autogluon_presets": AUTOGLUON_PRESETS,
        "autogluon_time_limit_sec": AUTOGLUON_TIME_LIMIT,
        "comparison_reference": "NB9 AutoGluon 600s",
    })

    if (
        METRICS_PATH.exists()
        and SUMMARY_PATH.exists()
        and LEADERBOARD_PATH.exists()
        and not FORCE_AUTOML_RETRAIN
    ):
        print("Loading cached NB11 AutoGluon artifacts...")

        metrics_nb11 = pd.read_csv(
            METRICS_PATH,
            parse_dates=["train_end", "val_start", "val_end"],
        )

        summary_nb11 = pd.read_csv(SUMMARY_PATH)
        leaderboard_nb11 = pd.read_csv(LEADERBOARD_PATH)

    else:
        print("Training AutoGluon extended-budget benchmark...")

        start_time = time.time()

        metrics_nb11, leaderboards_nb11 = evaluate_autogluon(
            df=advanced_train,
            features=advanced_features,
            folds=selected_folds,
            model_dir=AUTOGLUON_DIR,
            time_limit=AUTOGLUON_TIME_LIMIT,
            presets=AUTOGLUON_PRESETS,
        )

        total_runtime_sec = time.time() - start_time

        summary_nb11 = pd.DataFrame([{
            "model": "autogluon_tabular_extended_budget",
            "rmsle_mean": metrics_nb11["rmsle"].mean(),
            "rmsle_std": metrics_nb11["rmsle"].std(),
            "mae_mean": metrics_nb11["mae"].mean(),
            "mae_std": metrics_nb11["mae"].std(),
            "rmse_mean": metrics_nb11["rmse"].mean(),
            "r2_mean": metrics_nb11["r2"].mean(),
            "train_time_mean_sec": metrics_nb11["train_time_sec"].mean(),
            "inference_time_mean_sec": metrics_nb11["inference_time_sec"].mean(),
            "total_runtime_sec": total_runtime_sec,
            "time_limit_sec": AUTOGLUON_TIME_LIMIT,
            "n_folds": len(selected_folds),
        }])

        leaderboard_nb11 = pd.concat(leaderboards_nb11, ignore_index=True)

        metrics_nb11.to_csv(METRICS_PATH, index=False)
        summary_nb11.to_csv(SUMMARY_PATH, index=False)
        leaderboard_nb11.to_csv(LEADERBOARD_PATH, index=False)

    display(metrics_nb11)
    display(summary_nb11)
    display(leaderboard_nb11)

    for _, row in metrics_nb11.iterrows():
        fold = int(row["fold"])
        mlflow.log_metric(f"fold_{fold}_rmsle", row["rmsle"])
        mlflow.log_metric(f"fold_{fold}_mae", row["mae"])
        mlflow.log_metric(f"fold_{fold}_rmse", row["rmse"])
        mlflow.log_metric(f"fold_{fold}_r2", row["r2"])
        mlflow.log_metric(f"fold_{fold}_train_time_sec", row["train_time_sec"])
        mlflow.log_metric(f"fold_{fold}_inference_time_sec", row["inference_time_sec"])
        mlflow.log_param(f"fold_{fold}_best_model", row["best_model"])

    summary_row = summary_nb11.iloc[0]

    mlflow.log_metric("rmsle_mean", summary_row["rmsle_mean"])
    mlflow.log_metric("mae_mean", summary_row["mae_mean"])
    mlflow.log_metric("rmse_mean", summary_row["rmse_mean"])
    mlflow.log_metric("r2_mean", summary_row["r2_mean"])
    mlflow.log_metric("train_time_mean_sec", summary_row["train_time_mean_sec"])
    mlflow.log_metric("inference_time_mean_sec", summary_row["inference_time_mean_sec"])

Training AutoGluon extended-budget benchmark...

Starting AutoGluon benchmark - Fold 2


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.3
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025
CPU Count:          12
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       5.51 GB / 9.71 GB (56.7%)
Disk Space Avail:   934.62 GB / 1006.85 GB (92.8%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 3600s
AutoGluon will save models to "/home/donatocorbacio/projects/store-sales-project/artifacts/nb11_autogluon_extended_budget/autogluon_models/fold_2"
Train Data Rows:    2926044
Train Data Columns: 16
Label Column:       sales
Problem Type:       regression
Preprocessing data ...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:        

[1000]	valid_set's rmse: 181.553
[2000]	valid_set's rmse: 172.321
[3000]	valid_set's rmse: 167.702
[4000]	valid_set's rmse: 164.68
[5000]	valid_set's rmse: 162.65
[6000]	valid_set's rmse: 161.284
[7000]	valid_set's rmse: 159.946
[8000]	valid_set's rmse: 158.929
[9000]	valid_set's rmse: 157.993
[10000]	valid_set's rmse: 157.063


	-157.0468	 = Validation score   (-root_mean_squared_error)
	541.94s	 = Training   runtime
	6.67s	 = Validation runtime
Fitting model: LightGBM ... Training model for up to 3045.44s of the 3045.44s of remaining time.
	Fitting with cpus=6, gpus=0, mem=1.5/3.8 GB


[1000]	valid_set's rmse: 164.287
[2000]	valid_set's rmse: 158.366
[3000]	valid_set's rmse: 155.718
[4000]	valid_set's rmse: 154.056
[5000]	valid_set's rmse: 153.172
[6000]	valid_set's rmse: 152.976
[7000]	valid_set's rmse: 153.265


	-152.8617	 = Validation score   (-root_mean_squared_error)
	351.66s	 = Training   runtime
	3.13s	 = Validation runtime
Fitting model: RandomForestMSE ... Training model for up to 2690.43s of the 2690.43s of remaining time.
	To force training the model, specify the model hyperparameter "ag.max_memory_usage_ratio" to a larger value (currently 1.0, set to >=7.10 to avoid the error)
		To set the same value for all models, do the following when calling predictor.fit: `predictor.fit(..., ag_args_fit={"ag.max_memory_usage_ratio": VALUE})`
		Setting "ag.max_memory_usage_ratio" to values above 1 may result in out-of-memory errors. You may consider using a machine with more memory as a safer alternative.
	Not enough memory to train RandomForestMSE... Skipping this model.
Fitting model: CatBoost ... Training model for up to 2689.40s of the 2689.39s of remaining time.
	Fitting with cpus=6, gpus=0, mem=1.9/3.8 GB
	-148.3573	 = Validation score   (-root_mean_squared_error)
	1894.64s	 = Training   r

AutoGluon | Fold 2 | Best=WeightedEnsemble_L2 | RMSLE=0.591546 | MAE=55.781 | RMSE=191.539 | R2=0.977273 | Train time=3605.5s


,fold,train_end,val_start,val_end,n_train,n_val,train_time_sec,inference_time_sec,autogluon_time_limit,autogluon_presets,best_model,best_model_score_val,best_model_fit_time,best_model_pred_time_val,rmsle,mae,rmse,r2
0,2,2017-07-18,2017-07-19,2017-08-15,2926044,49896,3605.516391,20.79768,3600,medium_quality,WeightedEnsemble_L2,-145.373075,2986.691479,10.199927,0.591546,55.780798,191.538916,0.977273


,model,rmsle_mean,rmsle_std,mae_mean,mae_std,rmse_mean,r2_mean,train_time_mean_sec,inference_time_mean_sec,total_runtime_sec,time_limit_sec,n_folds
0,autogluon_tabular_extended_budget,0.591546,NaN,55.780798,NaN,191.538916,0.977273,3605.516391,20.79768,3628.243743,3600,1


,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order,fold
0,WeightedEnsemble_L2,-145.373075,root_mean_squared_error,10.199927,2986.691479,0.001086,0.026713,2,True,6,2
1,CatBoost,-148.357259,root_mean_squared_error,0.072676,1894.636215,0.072676,1894.636215,1,True,3,2
2,LightGBM,-152.861746,root_mean_squared_error,3.127443,351.656355,3.127443,351.656355,1,True,2,2
3,XGBoost,-156.227417,root_mean_squared_error,0.326719,198.434533,0.326719,198.434533,1,True,5,2
4,LightGBMXT,-157.046819,root_mean_squared_error,6.672003,541.937663,6.672003,541.937663,1,True,1,2
5,NeuralNetFastAI,-249.608401,root_mean_squared_error,0.252304,592.827241,0.252304,592.827241,1,True,4,2


## Load NB9 Reference Results

To evaluate the value of the longer AutoML budget, NB11 is compared against NB9.

NB9 is the reference AutoGluon benchmark using 600 seconds per fold.

In [8]:
nb9_metrics = pd.read_csv(
    NB9_METRICS_PATH,
    parse_dates=["train_end", "val_start", "val_end"],
)

nb9_summary = pd.read_csv(NB9_SUMMARY_PATH)
nb9_leaderboard = pd.read_csv(NB9_LEADERBOARD_PATH)

display(nb9_metrics)
display(nb9_summary)

,fold,train_end,val_start,val_end,n_train,n_val,train_time_sec,inference_time_sec,autogluon_time_limit,autogluon_presets,best_model,best_model_score_val,best_model_fit_time,best_model_pred_time_val,rmsle,mae,rmse,r2
0,1,2017-06-20,2017-06-21,2017-07-18,2876148,49896,603.996969,14.644537,600,medium_quality,WeightedEnsemble_L2,-232.503638,586.261508,8.113820,0.476525,53.084927,203.580231,0.976389
1,2,2017-07-18,2017-07-19,2017-08-15,2926044,49896,601.573003,14.101837,600,medium_quality,WeightedEnsemble_L2,-156.877041,587.524698,7.493559,0.475665,58.002139,198.500102,0.975591


,model,rmsle_mean,rmsle_std,mae_mean,mae_std,rmse_mean,r2_mean,train_time_mean_sec,inference_time_mean_sec,nb5_reference_rmsle,nb5_reference_mae,nb5_reference_rmse,nb5_reference_r2
0,autogluon_tabular,0.476095,0.000608,55.543533,3.476993,201.040167,0.97599,602.784986,14.373187,0.59988,68.447697,248.706697,0.965119


## Compare NB9 600s vs NB11 3600s

The comparison is done on the same validation fold used by NB11.

In [9]:
selected_fold_id = int(metrics_nb11.iloc[0]["fold"])

nb9_same_fold = nb9_metrics[nb9_metrics["fold"] == selected_fold_id].copy()
nb11_same_fold = metrics_nb11.copy()

comparison_rows = []

comparison_rows.append({
    "experiment": "NB9_AutoGluon_600s",
    "fold": selected_fold_id,
    "time_limit_sec": 600,
    "rmsle": nb9_same_fold["rmsle"].iloc[0],
    "mae": nb9_same_fold["mae"].iloc[0],
    "rmse": nb9_same_fold["rmse"].iloc[0],
    "r2": nb9_same_fold["r2"].iloc[0],
    "train_time_sec": nb9_same_fold["train_time_sec"].iloc[0],
    "inference_time_sec": nb9_same_fold["inference_time_sec"].iloc[0],
    "best_model": nb9_same_fold["best_model"].iloc[0],
})

comparison_rows.append({
    "experiment": "NB11_AutoGluon_3600s",
    "fold": selected_fold_id,
    "time_limit_sec": AUTOGLUON_TIME_LIMIT,
    "rmsle": nb11_same_fold["rmsle"].iloc[0],
    "mae": nb11_same_fold["mae"].iloc[0],
    "rmse": nb11_same_fold["rmse"].iloc[0],
    "r2": nb11_same_fold["r2"].iloc[0],
    "train_time_sec": nb11_same_fold["train_time_sec"].iloc[0],
    "inference_time_sec": nb11_same_fold["inference_time_sec"].iloc[0],
    "best_model": nb11_same_fold["best_model"].iloc[0],
})

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

,experiment,fold,time_limit_sec,rmsle,mae,rmse,r2,train_time_sec,inference_time_sec,best_model
0,NB9_AutoGluon_600s,2,600,0.475665,58.002139,198.500102,0.975591,601.573003,14.101837,WeightedEnsemble_L2
1,NB11_AutoGluon_3600s,2,3600,0.591546,55.780798,191.538916,0.977273,3605.516391,20.797680,WeightedEnsemble_L2


## Improvement Analysis

This section measures whether the longer AutoML budget produces a meaningful improvement.

For error metrics such as RMSLE, MAE and RMSE, lower is better.

For R², higher is better.

In [10]:
nb9_row = comparison_df[comparison_df["experiment"] == "NB9_AutoGluon_600s"].iloc[0]
nb11_row = comparison_df[comparison_df["experiment"] == "NB11_AutoGluon_3600s"].iloc[0]

improvement_summary = pd.DataFrame([{
    "fold": selected_fold_id,
    "rmsle_nb9_600s": nb9_row["rmsle"],
    "rmsle_nb11_3600s": nb11_row["rmsle"],
    "rmsle_absolute_improvement": nb9_row["rmsle"] - nb11_row["rmsle"],
    "rmsle_percent_improvement": ((nb9_row["rmsle"] - nb11_row["rmsle"]) / nb9_row["rmsle"]) * 100,

    "mae_nb9_600s": nb9_row["mae"],
    "mae_nb11_3600s": nb11_row["mae"],
    "mae_absolute_improvement": nb9_row["mae"] - nb11_row["mae"],
    "mae_percent_improvement": ((nb9_row["mae"] - nb11_row["mae"]) / nb9_row["mae"]) * 100,

    "rmse_nb9_600s": nb9_row["rmse"],
    "rmse_nb11_3600s": nb11_row["rmse"],
    "rmse_absolute_improvement": nb9_row["rmse"] - nb11_row["rmse"],
    "rmse_percent_improvement": ((nb9_row["rmse"] - nb11_row["rmse"]) / nb9_row["rmse"]) * 100,

    "r2_nb9_600s": nb9_row["r2"],
    "r2_nb11_3600s": nb11_row["r2"],
    "r2_absolute_improvement": nb11_row["r2"] - nb9_row["r2"],

    "train_time_nb9_600s": nb9_row["train_time_sec"],
    "train_time_nb11_3600s": nb11_row["train_time_sec"],
    "train_time_ratio": nb11_row["train_time_sec"] / nb9_row["train_time_sec"],
}])

display(improvement_summary)

,fold,rmsle_nb9_600s,rmsle_nb11_3600s,rmsle_absolute_improvement,rmsle_percent_improvement,mae_nb9_600s,mae_nb11_3600s,mae_absolute_improvement,mae_percent_improvement,rmse_nb9_600s,rmse_nb11_3600s,rmse_absolute_improvement,rmse_percent_improvement,r2_nb9_600s,r2_nb11_3600s,r2_absolute_improvement,train_time_nb9_600s,train_time_nb11_3600s,train_time_ratio
0,2,0.475665,0.591546,-0.115881,-24.362015,58.002139,55.780798,2.22134,3.829756,198.500102,191.538916,6.961186,3.506893,0.975591,0.977273,0.001682,601.573003,3605.516391,5.993481


## Leaderboard Inspection

This section checks whether the longer budget allowed AutoGluon to train additional model families or only improved the same LightGBM-based candidates.

In [11]:
print("NB9 leaderboard models:")
display(
    nb9_leaderboard[nb9_leaderboard["fold"] == selected_fold_id]
    .sort_values("score_val", ascending=False)
)

print("NB11 leaderboard models:")
display(
    leaderboard_nb11
    .sort_values("score_val", ascending=False)
)

NB9 leaderboard models:


,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order,fold
3,WeightedEnsemble_L2,-156.877041,root_mean_squared_error,7.493559,587.524698,0.000561,0.014050,2,True,3,2
4,LightGBMXT,-157.046819,root_mean_squared_error,7.434825,566.318337,7.434825,566.318337,1,True,1,2
5,LightGBM,-181.277652,root_mean_squared_error,0.058172,21.192311,0.058172,21.192311,1,True,2,2


NB11 leaderboard models:


,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order,fold
0,WeightedEnsemble_L2,-145.373075,root_mean_squared_error,10.199927,2986.691479,0.001086,0.026713,2,True,6,2
1,CatBoost,-148.357259,root_mean_squared_error,0.072676,1894.636215,0.072676,1894.636215,1,True,3,2
2,LightGBM,-152.861746,root_mean_squared_error,3.127443,351.656355,3.127443,351.656355,1,True,2,2
3,XGBoost,-156.227417,root_mean_squared_error,0.326719,198.434533,0.326719,198.434533,1,True,5,2
4,LightGBMXT,-157.046819,root_mean_squared_error,6.672003,541.937663,6.672003,541.937663,1,True,1,2
5,NeuralNetFastAI,-249.608401,root_mean_squared_error,0.252304,592.827241,0.252304,592.827241,1,True,4,2


## Cost-Benefit Decision Table

This table summarizes whether the longer AutoML budget is justified from an engineering perspective.

In [15]:
rmsle_improvement_pct = improvement_summary["rmsle_percent_improvement"].iloc[0]
mae_improvement_pct = improvement_summary["mae_percent_improvement"].iloc[0]
rmse_improvement_pct = improvement_summary["rmse_percent_improvement"].iloc[0]
train_time_ratio = improvement_summary["train_time_ratio"].iloc[0]

if rmsle_improvement_pct > 0 and mae_improvement_pct > 0:
    decision = "Broad improvement"
elif rmsle_improvement_pct < 0 and mae_improvement_pct > 0:
    decision = "Context-dependent trade-off"
else:
    decision = "Not justified"

cost_benefit_df = pd.DataFrame([{
    "experiment": "NB11_AutoGluon_3600s",
    "reference": "NB9_AutoGluon_600s",
    "fold": selected_fold_id,
    "rmsle_improvement_pct": rmsle_improvement_pct,
    "mae_improvement_pct": mae_improvement_pct,
    "rmse_improvement_pct": rmse_improvement_pct,
    "train_time_ratio": train_time_ratio,
    "engineering_decision": decision,
}])

display(cost_benefit_df)

,experiment,reference,fold,rmsle_improvement_pct,mae_improvement_pct,rmse_improvement_pct,train_time_ratio,engineering_decision
0,NB11_AutoGluon_3600s,NB9_AutoGluon_600s,2,-24.362015,3.829756,3.506893,5.993481,Context-dependent trade-off


## Save NB11 Comparison Artifacts

In [14]:
comparison_df.to_csv(ARTIFACTS_DIR / "nb11_vs_nb9_comparison.csv", index=False)
improvement_summary.to_csv(ARTIFACTS_DIR / "nb11_improvement_summary.csv", index=False)
cost_benefit_df.to_csv(ARTIFACTS_DIR / "nb11_cost_benefit_decision.csv", index=False)

print("Saved NB11 artifacts to:", ARTIFACTS_DIR.resolve())

Saved NB11 artifacts to: /home/donatocorbacio/projects/store-sales-project/artifacts/nb11_autogluon_extended_budget


## Final Conclusion

NB11 evaluates whether increasing the AutoGluon time budget from 600 seconds to 3600 seconds
improves model quality enough to justify the additional computational cost.

The experiment was intentionally controlled:
- same full dataset
- same advanced_v1 features
- same walk-forward validation logic
- same AutoGluon Tabular setup
- longer AutoML search budget

NB11 confirms that extending the AutoML search does not change the modeling paradigm:
the best solution remains a tree-based ensemble, consistent with NB9.

The longer search improves absolute error metrics (MAE, RMSE) and slightly improves R²,
but degrades RMSLE, indicating weaker relative calibration on lower-volume observations.

This means NB11 should not be interpreted as strictly better or worse than NB9.
It represents a context-dependent trade-off:

- NB9 remains the best default cost/performance configuration
- NB11 is the better option when reducing absolute forecast error matters more than training cost

From an engineering perspective, NB11 is best treated as an upper-bound benchmark:
useful to quantify the value of deeper AutoML search, but not the default operational choice.

## NB11 Whitebox Insight

NB11 does not improve by discovering a new modeling paradigm.

The winning model remains WeightedEnsemble_L2, confirming that the best-performing
solution is still a tree-based stacked ensemble, consistent with NB9.

What changes in NB11 is not the winner type, but the ensemble composition.

NB9 was dominated by LightGBM-style models:
- LightGBMXT
- LightGBM
- WeightedEnsemble_L2 on top

NB11 keeps the same ensemble paradigm, but the committee becomes more diversified:
- CatBoost (0.50)
- LightGBM (0.25)
- LightGBMXT (0.125)
- XGBoost (0.125)

This means the additional AutoML budget did not discover a fundamentally new model family.
Instead, it improved performance by building a richer ensemble of complementary tree boosters.

Interpretation:
- CatBoost becomes the dominant contributor
- LightGBM remains the main stabilizer
- LightGBMXT and XGBoost act as secondary refiners

The gain in NB11 is therefore best explained by ensemble diversification,
not by a change in model class.

In [16]:
whitebox_summary = pd.DataFrame([
    {
        "notebook": "NB9",
        "winner": "WeightedEnsemble_L2",
        "dominant_components": "LightGBMXT + LightGBM",
        "ensemble_type": "LightGBM-dominant",
        "interpretation": "Simpler, lighter tree ensemble"
    },
    {
        "notebook": "NB11",
        "winner": "WeightedEnsemble_L2",
        "dominant_components": "CatBoost + LightGBM + LightGBMXT + XGBoost",
        "ensemble_type": "Diversified tree ensemble",
        "interpretation": "Richer ensemble, better absolute error"
    }
])

display(whitebox_summary)

,notebook,winner,dominant_components,ensemble_type,interpretation
0,NB9,WeightedEnsemble_L2,LightGBMXT + LightGBM,LightGBM-dominant,"Simpler, lighter tree ensemble"
1,NB11,WeightedEnsemble_L2,CatBoost + LightGBM + LightGBMXT + XGBoost,Diversified tree ensemble,"Richer ensemble, better absolute error"


In [17]:
# Export NB11 fold-level predictions for NB12

from autogluon.tabular import TabularPredictor

PREDICTIONS_PATH = ARTIFACTS_DIR / "nb11_fold_predictions.csv"

selected_fold_id = 2
fold_info = selected_folds[0]

val_mask = (
    (advanced_train["date"] >= fold_info["val_start"]) &
    (advanced_train["date"] <= fold_info["val_end"])
)

val_df = advanced_train.loc[val_mask].copy()

predictor_path = AUTOGLUON_DIR / f"fold_{selected_fold_id}"
predictor = TabularPredictor.load(str(predictor_path))

val_df["prediction"] = predictor.predict(val_df[advanced_features])
val_df["prediction"] = val_df["prediction"].clip(lower=0)
val_df["fold"] = selected_fold_id

nb11_predictions = val_df[
    ["fold", "date", "store_nbr", "family", "sales", "prediction", "onpromotion"]
].copy()

nb11_predictions.to_csv(PREDICTIONS_PATH, index=False)

print("Saved NB11 predictions to:", PREDICTIONS_PATH.resolve())
display(nb11_predictions.head())

Saved NB11 predictions to: /home/donatocorbacio/projects/store-sales-project/artifacts/nb11_autogluon_extended_budget/nb11_fold_predictions.csv


,fold,date,store_nbr,family,sales,prediction,onpromotion
2950992,2,2017-07-19,1,0,7.0,10.533682,0
2952774,2,2017-07-20,1,0,4.0,8.521036,0
2954556,2,2017-07-21,1,0,10.0,9.044243,0
2956338,2,2017-07-22,1,0,8.0,0.000000,0
2958120,2,2017-07-23,1,0,0.0,0.000000,0
